# 06_compare_base_tuned: Did the Fine-Tune Work?

[Open in Colab](https://colab.research.google.com/github/Utkarsh-09/AI_GURU_labs/blob/main/notebooks/06_compare_base_tuned.ipynb)

**Session:** Day 2, S12 — Did it work (45-minute block)
**Expected runtime:** **25 minutes**, almost all of it yours: three small TODOs, reading one table, and the rubric conversation. The machine part is short. Measured on a Colab free-tier **T4** on 2026-09-21: Ollama install **56 s**, model pull **28 s**, warm-up 3 s with 100% of the model in GPU memory, then **17 s** of model time for the 20 base tickets and **21 s** for the 20 tuned ones (about 0.8 s a ticket). The whole sitting, including filling in the three TODOs by hand, took **4.2 minutes** from the settings cell to the final cell.
**Needs:** **Google Colab on the free T4 GPU runtime** — the same one as notebook 05; never Pro. That is the only supported place to run this lab. The notebook installs and starts Ollama itself, at the one pinned release (0.12.10) that can still load a LoRA adapter; a Colab *CPU* runtime is too slow (a ticket takes longer than the endpoint's timeout) and the notebook stops with instructions if it finds itself on one. **A laptop is not a supported path for the tuned model:** current Ollama releases refuse adapters, so the `tuned` endpoint cannot be created there. No API key. Reads `data/eval/heldout_20.jsonl`, `scripts/run_eval.py`, `scripts/register_adapter.py`, and your adapter from notebook 05 — **or, if that is missing or broken, `checkpoints/adapter_prebaked/`. The notebook prints which one it is using. It never swaps silently.**
**A correct result looks like:** the final cell prints `COMPARISON READY`, names the adapter in use, and shows one table: the same 20 held-out tickets, one column for the untuned model and one for the same model with the adapter. With the pre-baked adapter on Ollama 0.12.10 the columns read about: schema-valid **10/20 → 20/20**, `routing_queue` **5/20 → 16/20**, whole record **2/20 → 4/20**, urgency **5/20 → 7/20**, invented values **2 → 0** (the Colab T4 run retained in the solution). The base column moves by a few tickets from machine to machine and sitting to sitting (measured: schema-valid anywhere from 9/20 to 13/20 — the untuned model's replies are not stable even at temperature 0), while the tuned column of this adapter was identical every time; your own adapter will differ by a ticket or three. The tuned model is **not** right everywhere — urgency stays poor, and on a couple of tickets it does *worse* than the untuned model. The notebook shows you those tickets on purpose.

> All data in this lab is synthetic. No real OQ material anywhere.

---
**The plan.** Load the 20 held-out tickets → decide which adapter is being scored → get Ollama answering → score the base model with **one command** → score the tuned model with **the same command and one word changed** (**TODO 1**) → merge the two into **one table** (**TODO 2**) → read four tickets side by side, chosen by rule → write down what you conclude (**TODO 3**).

**If the runtime disconnects:** reconnect and *Run all*. Every model reply was saved to your checkpoint folder the moment it arrived, so a finished run is re-scored from disk in a second and a half-finished run continues at the ticket where it stopped. Only Ollama itself has to be set up again on a new Colab runtime.

**Why this cell:** one notebook has to run in two places — Colab during
the program, and a local machine as the fallback. This first code cell
detects which one it woke up in, mounts Google Drive on Colab so
checkpoints survive a disconnect, and sets the three variables every
later cell can rely on: `IN_COLAB`, `REPO_ROOT`, `CHECKPOINT_DIR`.

In [ ]:
# Environment detection: Colab vs local. Sets IN_COLAB, REPO_ROOT, CHECKPOINT_DIR.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# The repo URL participants clone in Colab. Set once, here.
REPO_URL = "https://github.com/Utkarsh-09/AI_GURU_labs.git"

if IN_COLAB:
    # Drive first: checkpoints survive a runtime disconnect.
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_ROOT = Path("/content/oq-advanced-ai")
    if not REPO_ROOT.exists():
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/oq-advanced-ai-checkpoints")
else:
    # Local: find the repo root by walking up until BUILD_SPEC.md appears.
    here = Path.cwd()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / "BUILD_SPEC.md").exists()),
        here,
    )
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "local"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Make repo modules importable: config.endpoints, notebooks/utils.py
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

print(f"Environment : {'Colab' if IN_COLAB else 'local'}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Checkpoints : {CHECKPOINT_DIR}")

**Why this cell:** the eval harness needs two Python packages, `requests` (to call the model) and `jsonschema` (to check a reply against the ticket schema). Colab already ships both at exactly the pinned versions, so this line is a no-op there and costs a second. Ollama is not a Python package — it gets its own cell below.

In [ ]:
# Pinned installs — versions match requirements.txt. Colab only;
# local machines installed requirements.txt during setup.
if IN_COLAB:
    %pip install -q requests==2.32.4 jsonschema==4.26.0
print("Install cell done.")

**Why this cell:** everything that decides *what gets compared* sits in one place. `MODEL_NAME` and `RUN_NAME` are the same two names you set in notebook 05 — together they say which adapter is yours. `USE_PREBAKED = True` is the switch for a group whose training finished but went wrong (say, a learning-rate experiment that blew up): it scores the pre-baked adapter instead, and says so. All results go to `eval_dir`, inside your checkpoint folder — on Google Drive when you are in Colab.

In [ ]:
import time

import compare_utils
import ollama_utils
import utils

MODEL_NAME = "llama3.2-1b"     # as in notebook 05
RUN_NAME = "run1"              # as in notebook 05
USE_PREBAKED = False           # True = score the pre-baked adapter even if yours exists

heldout_path = REPO_ROOT / "data" / "eval" / "heldout_20.jsonl"
run_eval_script = REPO_ROOT / "scripts" / "run_eval.py"
register_script = REPO_ROOT / "scripts" / "register_adapter.py"
eval_dir = CHECKPOINT_DIR / "eval"

sitting_started = time.time()

print(f"looking for : the adapter of notebook 05 run '{MODEL_NAME}_{RUN_NAME}'")
print(f"questions   : {heldout_path}")
print(f"results to  : {eval_dir}")

**Why this cell (step 1):** know the exam before reading the marks. These 20 tickets were set aside before the train/validation split was made; no model in this lab has trained on them, and `scripts/build_dataset.py` refuses any ticket that has a lookalike elsewhere in the corpus. Note how small and how lopsided the set is — five `access` tickets, two `telecom`. **One ticket is five percentage points.** Keep that in your head for the rest of the lab.

In [ ]:
import dataset_utils

heldout_pairs = dataset_utils.load_jsonl(heldout_path)
category_counts = dataset_utils.count_record_field(heldout_pairs, "category")
urgency_counts = dataset_utils.count_record_field(heldout_pairs, "urgency")

print(f"{len(heldout_pairs)} held-out tickets")
print(f"  category : {category_counts}")
print(f"  urgency  : {urgency_counts}")
print()
first_pair = heldout_pairs[0]
print(f"--- {first_pair['ticket_id']}: what the model is sent ---")
print(dataset_utils.pair_user_text(first_pair))
print("--- what a perfect reply looks like ---")
print(dataset_utils.pair_completion_text(first_pair))

**Why this cell (step 2) — which adapter?** A score is worthless if you cannot say what was scored. The helper looks for *your* adapter first: in your run folder (on Drive, so it survives a disconnect), then in the repo's `checkpoints/` folder where notebook 05 also copied it. It does not just check that a folder exists — it checks the weights file is complete, because a runtime that died while saving leaves a cut-off file behind. Only if nothing of yours is usable does it fall back to `checkpoints/adapter_prebaked/`, the adapter trained with this same notebook on a free T4 before the program. **Read the `ADAPTER IN USE` line.** If it says pre-baked, the table below is still a real result — it is just not *your* result, and you should say so when you present it.

In [ ]:
adapter = compare_utils.find_adapter(CHECKPOINT_DIR, REPO_ROOT, MODEL_NAME, RUN_NAME,
                                     use_prebaked=USE_PREBAKED)

print("Looked for an adapter in this order:")
for entry in adapter["checked"]:
    print(f"  {entry['path']}")
    print(f"      -> {entry['verdict']}")
print()

if adapter["source"] == "yours":
    print("ADAPTER IN USE: YOURS - the one you trained in notebook 05")
else:
    print("ADAPTER IN USE: THE PRE-BAKED ONE - NOT an adapter you trained")
    print("  (nothing usable of yours was found above; the lab carries on with the insurance copy)")
print(f"  folder      : {adapter['path']}")
print(f"  fingerprint : {adapter['fingerprint']}   (first 8 digits of the sha256 of the weights)")
print(f"  sits on     : {adapter['ollama_base']}   <- so THAT is the base model it must be compared with")
if adapter["training"] is not None:
    print(f"  trained     : {adapter['training']['steps_done']} of {adapter['training']['steps_planned']} steps, "
          f"{adapter['training']['training_minutes']:.1f} minutes")

utils.save_json(eval_dir, "06_adapter_in_use", adapter)

**Why this cell:** both models are served the way your integration would call them — over HTTP, by Ollama, through `config/endpoints.py`. The helper downloads **one pinned Ollama release**, 0.12.10 — never "latest" — starts `ollama serve` in the background and waits until it answers. The pin is not fussiness: turning an adapter into a servable model is a feature current Ollama releases have removed (`LoRA adapters are no longer supported`), so on any other version there is no tuned model to score. That is also why this lab is **Colab T4 only**: Colab is where the notebook controls the version. On a laptop the cell still runs, says plainly that you are off the supported path, and tells you whether the Ollama it found can work at all.

It checks the GPU **first**. A Colab **CPU** runtime has only two cores, and on two cores a ticket takes longer than the endpoint's 120-second timeout. So on Colab without a GPU the cell stops at once and says what to do, instead of downloading 1.9 GB and failing twenty minutes later. Then it pulls the base model *the adapter sits on*: an adapter is a set of corrections to one specific model's weights, so comparing it with any other model would compare two things at once. The warm-up line at the end is your early warning: if it says many seconds, or `0%` in GPU memory, fix that before scoring anything.

In [ ]:
gpu_name = ollama_utils.gpu_name()
print(f"NVIDIA GPU: {gpu_name or 'none found'}")

if IN_COLAB and gpu_name is None:
    print("This Colab runtime has no GPU, and its two CPU cores are too slow for this lab:")
    print("measured on a 2-core Linux machine, one ticket takes over 120 seconds, which the")
    print("endpoint counts as a timeout - the eval would stop with 'endpoint is not answering'.")
    print("  1. Runtime > Change runtime type > T4 GPU (free tier), then Run all. Nothing is lost.")
    print("  2. GPU quota used up? Join a group that has a GPU, or ask the facilitator for the")
    print("     pre-baked table. A laptop is NOT the way out: see the header of this notebook.")
    raise RuntimeError("No GPU in this Colab runtime - see the two options printed above.")

server = ollama_utils.ensure_server(IN_COLAB, log_dir=REPO_ROOT / "eval_runs")

if not IN_COLAB:
    print("LOCAL RUN - NOT A SUPPORTED PATH FOR THIS LAB. Supported: Google Colab, free T4 GPU runtime.")
    if server["version"] == ollama_utils.OLLAMA_VERSION:
        print(f"  This machine happens to have Ollama {server['version']}, the pinned release, so it can work.")
    else:
        print(f"  This machine has Ollama {server['version']}. Only {ollama_utils.OLLAMA_VERSION} is known to load")
        print("  an adapter; current releases refuse it, and the register cell below will then stop.")
        print("  Do not reinstall Ollama in the room - open this notebook in Colab instead.")
pull_result = ollama_utils.ensure_model(adapter["ollama_base"])
first_reply = ollama_utils.warm_up(adapter["ollama_base"])

print(f"Ollama {server['version']} answering at {server['url']}"
      f"  ({'started by this cell' if server['started_here'] else 'was already running'})")
print(f"base model {adapter['ollama_base']}: {pull_result}")
print(f"warm-up: loaded and answered in {first_reply['seconds']} s; "
      f"{first_reply['gpu_share']:.0%} of the model is in GPU memory")

**Why this cell:** the `tuned` endpoint is nothing more than an Ollama model with a name. `scripts/register_adapter.py` writes a three-line Modelfile — `FROM` the base model, `ADAPTER` your folder, a context size — and runs `ollama create`. It prints the Modelfile so you can see there is no trick. The two environment variables are how `config/endpoints.py` learns which model names the words `local` and `tuned` stand for in this lab. If registration fails the cell stops here: better no table than a table that scored some *older* model still registered under the same name.

In [ ]:
TUNED_MODEL_NAME = "oq-ticket-tuned"

os.environ["OLLAMA_MODEL"] = adapter["ollama_base"]    # what the endpoint "local" means below
os.environ["TUNED_MODEL"] = TUNED_MODEL_NAME           # what the endpoint "tuned" means below

register_command = [sys.executable, register_script, "--adapter", adapter["path"], "--name", TUNED_MODEL_NAME]
register_exit_code = compare_utils.run_command(register_command)
assert register_exit_code == 0, "The adapter was not registered - read the message above. Nothing was scored."

print()
print(f"models on this server: {ollama_utils.list_models()}")

**Why this cell (step 3, milestone 1) — the one command:** this is the whole eval, exactly as you would type it in a terminal; the cell only runs it and shows the output. `--endpoint local` is the untuned model. `--out` puts the result files in your checkpoint folder. `--resume` reuses replies already saved under this `--run-id` — that is what makes a disconnect cheap — and it is safe here because the run id names the model, so a saved reply can only ever belong to this model.

The report is long on purpose. For now read sections **1** (format) and **3** (whole record), and the warning block at the end. Sections 6 and 7 come back in the rubric conversation.

In [ ]:
base_run_id = f"06_base_{adapter['model_key']}"

base_command = [
    sys.executable, run_eval_script,
    "--dataset", heldout_path,
    "--endpoint", "local",
    "--label", "base",
    "--run-id", base_run_id,
    "--out", eval_dir,
    "--resume",
]
base_exit_code = compare_utils.run_command(base_command)
assert base_exit_code == 0, "The base run did not complete - read the message above, fix it, run this cell again."

**Why this cell (step 4, milestone 2) — TODO 1, the same command:** now the tuned model. Nothing about the harness changes: same tickets, same prompt, same scoring rules, same temperature. **The only thing that differs between measuring the base model and measuring your fine-tune is the name of the endpoint** — which is the point of having an endpoint config at all, and the reason a comparison made this way can be trusted. The run id carries the adapter's fingerprint, so replies saved for one adapter are never reused for another.

In [ ]:
# ── TODO 1 ─────────────────────────────────────────────────────────
# Score the tuned model: the SAME command as the base run, two words changed.
# Hint: config/endpoints.py knows three endpoint names: "local", "hosted", "tuned".
# The label is the column heading in the table - pick one that says what the column is.
tuned_endpoint = ...   # <- replace the ... with the endpoint name
tuned_label = ...      # <- replace the ... with a column heading
# ───────────────────────────────────────────────────────────────────

assert tuned_endpoint is not ... and tuned_label is not ..., "TODO 1 is not filled in yet"

tuned_run_id = f"06_tuned_{adapter['source']}_{adapter['fingerprint']}"

tuned_command = [
    sys.executable, run_eval_script,
    "--dataset", heldout_path,
    "--endpoint", tuned_endpoint,
    "--label", tuned_label,
    "--run-id", tuned_run_id,
    "--out", eval_dir,
    "--resume",
]
tuned_exit_code = compare_utils.run_command(tuned_command)
assert tuned_exit_code == 0, "The tuned run did not complete - read the message above, fix it, run this cell again."

**Why this cell (step 5, milestone 3) — TODO 2, the one table:** every run wrote a `<run_id>_summary.json` holding each number in its report. `--compare` merges summary files into one table, and **refuses** if the runs did not answer exactly the same questions — a table comparing different exams is worse than no table. Four kinds of number are kept apart and never blended into one score: FORMAT, per-FIELD accuracy, whole RECORD, and INVENTED values.

**Read it like a sceptic — especially if the tuned column wins every row.** A clean sweep is a reason to check the exam before trusting the marks: (1) *Did the model see these tickets?* No — they were held out before the split, and notebook 04 measured 0 overlap. (2) *Which wins are bigger than two tickets?* At n = 20 anything smaller is noise; expect the large, real gaps in format, `routing_queue` and `requested_action` — the house conventions a prompt cannot fully convey — and expect `urgency` and the whole-record row to stay poor. (3) *Does it win on every ticket?* Read the PER TICKET block: there are tickets where the tuned number is **lower**. The next cell shows you one.

One more thing the table cannot show: the untuned model is not even consistent with *itself*. Run this lab on a Windows laptop and in a Linux container — same Ollama release, same tickets, same prompt, temperature 0 — and **12 of the base model's 20 replies differ**; its schema-valid count was 13/20 on one machine and 9/20 on the other, and 10/20 on a Colab T4. A small untuned model has many near-tied choices, and rounding differences between machines are enough to tip them. The tuned model's 20 replies were **identical, byte for byte**, on the two machines where they could be compared, and its scores on the T4 matched them ticket for ticket. So the base column moves by a few tickets between machines — do not build an argument on one of its tickets — and notice that consistency is something the fine-tune bought that no row of the table reports.

In [ ]:
base_summary_file = eval_dir / f"{base_run_id}_summary.json"

# ── TODO 2 ─────────────────────────────────────────────────────────
# Point the comparison at the tuned run's summary file.
# Hint: it is named like the base one, one line up - with the tuned run's id.
tuned_summary_file = ...   # <- replace the ... with the path
# ───────────────────────────────────────────────────────────────────

assert tuned_summary_file is not ..., "TODO 2 is not filled in yet"

comparison_dir = eval_dir / "06_base_vs_tuned"

compare_command = [
    sys.executable, run_eval_script,
    "--compare", base_summary_file, tuned_summary_file,
    "--out", comparison_dir,
]
compare_exit_code = compare_utils.run_command(compare_command)
assert compare_exit_code == 0, "No table - read the message above."

**Why this cell (step 6) — four tickets, chosen by rule:** a table tells you *how often*; only tickets tell you *how*. To keep anyone — including the people who built this lab — from showing only the flattering ones, the tickets are picked by fixed rules (`compare_utils.pick_examples`): the first ticket whose **format** tuning rescued; the biggest **content** gain among tickets both models answered readably; the ticket where the tuned model did **worse** than the untuned one, if there is one; and the tuned model's **worst remaining** ticket. `<-- X` marks a wrong field. Look hardest at the last two.

In [ ]:
base_run = compare_utils.load_run(eval_dir, base_run_id)
tuned_run = compare_utils.load_run(eval_dir, tuned_run_id)
picks = compare_utils.pick_examples(base_run, tuned_run)

ticket_texts = {}
for pair in heldout_pairs:
    ticket_texts[pair["ticket_id"]] = dataset_utils.pair_user_text(pair)

for pick in picks:
    ticket_text = ticket_texts.get(pick["item_id"], "")
    print(compare_utils.render_example(pick, ticket_text, base_run, tuned_run))

**Why this cell (step 7) — TODO 3, what do you conclude?** Numbers do not decide anything until somebody writes down what they mean. This is the tally sheet from `data/eval/rubric.md`, as a dict, saved next to your results. Rules of the rubric: **counts, not adjectives**; four measurements stay four, no blended "quality score"; a 5-point gap is one ticket; and nothing you decide here changes a machine number.

- **H1 — urgency.** In the *tuned* report above, section 6: is there a `PATTERN` line? The system prompt never defines the four urgency levels. Whose job is it to teach the model your convention — the prompt, or the training data?
- **H2 — requested_action.** Section 7 of the tuned report lists clauses that scored below 0.50. For each: given only this clause and the routing queue, would a desk agent start the right work? YES or NO.
- **Fix first.** One thing, and the tool you would reach for: `prompt`, `tuning`, `retrieval`, or a `validator` in front of the downstream system.

In [ ]:
# ── TODO 3 ─────────────────────────────────────────────────────────
# Fill in every ... from the table, the two reports and the four tickets above.
# Hint: copy counts straight from the table, base first: "12/20 -> 20/20".
reflection = {
    "schema_valid": ...,       # FORMAT row, base -> tuned
    "whole_record": ...,       # RECORD row, base -> tuned
    "invented_values": ...,    # INVENTED row, base -> tuned
    "helped_most": ...,        # the field tuning moved most, with its counts
    "not_fixed": ...,          # a field that is still poor after tuning, with its counts
    "tuned_did_worse_on": ...,  # ticket ids from the PER TICKET block where the tuned number is lower
    "h1_pattern": ...,         # "over" | "under" | "mixed" | "none"   (tuned report, section 6)
    "h1_whose_job": ...,       # "prompt" | "training data"
    "h2_adequate": ...,        # e.g. "3 of 4"                         (tuned report, section 7)
    "fix_first": ...,          # one sentence
    "fix_with": ...,           # "prompt" | "tuning" | "retrieval" | "validator"
}
# ───────────────────────────────────────────────────────────────────

unfilled = [key for key, value in reflection.items() if value is ...]
assert unfilled == [], f"TODO 3 is not filled in yet: {unfilled}"

reflection["adapter_in_use"] = f"{adapter['source']} {adapter['fingerprint']}"
utils.save_json(eval_dir, "06_reflection", reflection)

**Why this cell:** the declared result, in one block, so "done" can be checked from across the room — and so the line saying *which adapter* was scored travels with the numbers. The same two summary files are what Day 4 (S19) puts next to a third column, base + retrieval, without re-running anything here.

In [ ]:
comparison = utils.load_json(comparison_dir, "comparison")
headline_lines = compare_utils.headline(comparison)
sitting_minutes = (time.time() - sitting_started) / 60

print("COMPARISON READY")
if adapter["source"] == "yours":
    print(f"  adapter in use : YOURS  ({adapter['fingerprint']})  - trained in notebook 05")
else:
    print(f"  adapter in use : PRE-BAKED  ({adapter['fingerprint']})  - not trained in this room")
print(f"  base model     : {adapter['ollama_base']} - the model the adapter sits on")
print(f"  questions      : {comparison['n_items']} held-out tickets, fingerprint {comparison['dataset_sha256']}")
print()
for line in headline_lines:
    print(line)
print()
print(f"  adapter folder : {adapter['path']}")
print(f"  the table      : {comparison_dir / 'comparison.txt'}")
print(f"  your notes     : {eval_dir / '06_reflection.json'}")
print(f"  this sitting   : {sitting_minutes:.1f} minutes from the settings cell to here")
print("  remember       : 20 tickets. One ticket is 5 points. Counts, not percentages.")